**BARNES MAZE ANALYSIS FOR DEEPLABCUT DATA**

---

last update: v.3 (4/22/26)

In [ ]:
## IMPORT TOOLBOXES

import pandas as pd
from pathlib import Path
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import cv2
import math

In [ ]:
# SET DIRECTORY

os.chdir("/path/to/your/project)

In [ ]:
## IMPORT CUSTOM BARNES MAZE SCRIPT
## make sure "barnes_maze_analysis_v3.py" is located in directory folder

import barnes_maze_analysis_v3 as BMA

In [ ]:
## EXTRACT COORDINATE DATA FOR ALL HOLES AND FOR TABLE
# creates 3 .csv files in your directory folder: "hole_coords.csv", "table_coords.csv", & "coord_summary_values.csv"

## change "DAY#" labels as needed for your project & provde file paths for DLC csv files of hole/table coordinates
coord_files = {
    "DAY2": "/path/to/your/Day2coordFile", 
    "DAY3": "/path/to/your/Day3coordFile",
    "DAY4": "/path/to/your/Day4coordFile",
    "DAY5": "/path/to/your/Day5coordFile",
    "DAY6": "/path/to/your/Day6coordFile",
}

## RUN THE FIRST PART OF THE ANALYSIS
my_coords = BMA.CoordsDataset(
    coord_files_dict = coord_files, 
    exp_days = (2,6),
    table_size_cm = 90,
    num_holes = 20,
    hole_entry_zone_cm = 10
)


In [ ]:
## RUN THE SECOND PART OF THE ANALYSIS
## a popup will prompt you to choose the file location of your DLC xy-data files (.csv's) & the output label and save location for results .csv 
## rois, cm_per_pixel, and hole_entry_diameter are set to call from class attributes calculated in previous block, but can be customized 

BMA.analyze(rois = my_coords.hole_coords_dict, 
                fps = 30.0, 
                cm_per_pixel = my_coords.coords_summary["px_to_cm_conversion"],
                rounding_digits = 4,
                x_shift = 100, ## optional
                y_shift = 10, ## optional
                target_hole = "H1",
                hole_entry_diameter = my_coords.coords_summary["hole_entry_zone_px"],
                quad_diameter = 144, ## optional
                quad_centers = ("H1", "H6", "H11", "H16") ## optional
)

**VISUAL CHECK OF HOLE / QUADRANT LOCATIONS**

---

In [ ]:
## can use the following scripts to check that your hole locations and quadrant placements are accurate relative to original video proportions
## can also use to visualize buffer zone size around holes to determine if entry zone needs to be adjusted
## you can use the quadrant visualization to figure out the size of your quadrants, as it's difficult to predict this on behalf of the user

## TIP: for the video_still file to overlay holes onto, just use an video_still still from your DLC project folder (you already have stills from labeling in Napari)


*VISUALIZING HOLE COORDINATES WITH BUFFER ZONES*

---


In [ ]:
## DATA OPTIONS (ADJUSTABLE)

video_still = cv2.imread("./DAY2.png") ## YOUR IMAGE PATH
day = "DAY2" ## CHANGE TO YOUR DESIRED DAY
target_hole = "H1" ## ESCAPE HOLE LOCATION
hole_radius = round(my_coords.coords_summary["hole_entry_zone_px"] / 2) ## USES DEFAULT DIAMETER FROM USER INPUT ABOVE; CAN CHANGE TO ANY NUMBER
table_center = my_coords.table_center_coods_dict['DAY2_center_coords'] ## CHANGE < DAY2 > IN "DAY2_center_coords" TO YOUR DESIRED DAY
table_radius = round(my_coords.coords_summary["table_size_px"] / 2) ## USES DEFAULT DIAMETER FROM USER INPUT ABOVE

############################## Don't need to ajust anything below this (inherits from objects set in DATA OPTIONS (ADJUSTABLE)

## GENERATE IMAGE
day_dict = my_coords.hole_coords_dict[day]
for coords in day_dict.values():
    cv2.circle(video_still, coords, hole_radius, (0,0,255), 1) ## cv2.circle(video_stillFile:str, (center_x:int, center_y:int), radius:int, [RGBcolor#], lineThickness:int)
## ARENA OUTLINE
cv2.circle(video_still, table_center, table_radius, [0, 0, 255], 1)
## OVERRIDE HOLE COLOR FOR TARGET HOLE (ASSUMES "H1")
cv2.circle(video_still, day_dict[f"{day}_{target_hole}"], hole_radius, (0,255,0), 1)
##LABEL IMAGE
title = f"{day} - Arena Setup"
cv2.putText(video_still, title ,(20,50), 0, .7, (255,255,255),2) ## cv2.putText(video_still, text:str, x_y_location:tuple, font_type:int, font_multiplier:float, color:tuple, font_thickness:int)

## PRINT TO NOTEBOOK
%matplotlib inline
plt.imshow(cv2.cvtColor(video_still, cv2.COLOR_BGR2RGB))
plt.show()


In [ ]:
## SAVE OVERLAY IMAGE TO PROJ. FOLDER

hole_overlay = cv2.imwrite("hole_overlay.jpg", video_still)

*VISUALIZING QUAD COORDINATES*

---

In [ ]:
## DATA OPTIONS (ADJUSTABLE)
video_still = cv2.imread("./DAY2.png") ## YOUR IMAGE PATH
day = "DAY2" ## CHANGE TO DESIRED DAY
quad_centers = ("H1", "H6", "H11", "H16") ## CHANGE EACH "H#" TO DESIRED CENTERS OF QUADS, IN ORDER OF QUAD1 THROUGH QUAD4
quad_radius = 140 ## QUADRANT RADIUS IN PIXELS; CHANGE TO DESIRED RADIUS
table_center = my_coords.table_center_coods_dict['DAY2_center_coords'] ## CHANGE < DAY2 > IN "DAY2_center_coords" TO YOUR DESIRED DAY
table_radius = round(my_coords.coords_summary["table_size_px"] / 2) ## USES DEFAULTS FROM USER INPUT ABOVE

############################## Don't need to ajust anything below this (inherits from objects set in DATA OPTIONS (ADJUSTABLE)

## CREATE FILTERED DICTIONARY
day_dict = my_coords.hole_coords_dict['DAY2']
quad_dict = {key: value for key, value in day_dict.items() if key.endswith(quad_centers)}

## GENERATE IMAGE
for coords in quad_dict.values():
    cv2.circle(video_still, coords, quad_radius, (0,0,255), 1) ## cv2.circle(video_stillFile:str, (center_x:int, center_y:int), radius:int, [RGBcolor#], lineThickness:int)
## ARENA OUTLINE
cv2.circle(video_still, table_center, table_radius, [0, 0, 255], 1)
## OVERRIDE HOLE COLOR FOR TARGET QUAD (ASSUMES CENTER OF TARGET QUAD IS HOLE 1) 
cv2.circle(video_still, day_dict[f"{day}_{quad_centers[0]}"], quad_radius, (0,255,0), 1)
##LABEL IMAGE
title = f"{day} - Arena Setup"
cv2.putText(video_still, title ,(20,50), 0, .7, (255,255,255),2) ## cv2.putText(video_still, text:str, x_y_location:tuple, font_type:int, font_multiplier:float, color:tuple, font_thickness:int)

## PRINT TO NOTEBOOK
%matplotlib inline
plt.imshow(cv2.cvtColor(video_still, cv2.COLOR_BGR2RGB))
plt.show()


In [ ]:
## SAVE OVERLAY IMAGE TO PROJ. FOLDER

quad_overlay = cv2.imwrite("quad_overlay.jpg", video_still)